# Exploratory Data Analysis


Dataset Features

| Feature               | Description                                                                                                    |
| --------------------- | -------------------------------------------------------------------------------------------------------------- |
| VendorID              | TPEP Provider Code (1. Creative Mobile Technologies, 2. VeriFone Inc.)                                         |
| tpep_pickup_datetime  | Meter engaged timestamp                                                                                        |
| tpep_dropoff_datetime | Meter disengaged timestamp                                                                                     |
| passenger_count       | Number of passengers                                                                                           |
| trip_distance         | Trip distance in miles                                                                                         |
| pickup_longitude      | Meter engaged longitude                                                                                        |
| pickup_latitude       | Meter engaged latitude                                                                                         |
| RateCodeID            | Rate code for type of ride (1. Standard, 2. JFK, 3. Newark, 4. Nassau or Westchester, 5. Negotiated, 6, Group) |
| store_and_fwd_flag    | Flag if trip record was held in memory temporarily due to lack of server connection                            |
| dropoff_longitude     | Meter disengaged longitude                                                                                     |
| dropoff_latitude      | Meter disengaged latitude                                                                                      |
| payment_type          | Mode of payment (1. Card, 2. Cash, 3. No charge, 4. Dispute, 5. Unknown, 6. Voided)                            |
| fare_amount           | Time and distance fare calculated by meter                                                                     |
| extra                 | Miscelleanous extras and surchanges (0.5 and 1 rush hour and overnight charges)                                |
| mta_tax               | Automatic 0.5 MTA tax based on metered rate                                                                    |
| tip_amount            | Tip amount (only for credit cards)                                                                             |
| tolls_amount          | Amount of tolls in trip                                                                                        |
| improvement_surcharge | 0.3 improvement surcharge assessed trips after flag drop                                                       |
| total_amount          | Total amount (excludes cash tips)                                                                              |


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
spark = (
    SparkSession.builder.config("spark.executor.memory", "10g")
    .config("spark.driver.memory", "10g")
    .appName("exploratory_data_analysis")
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/23 18:27:11 WARN Utils: Your hostname, legion-5i, resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlp0s20f3)
25/10/23 18:27:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/23 18:27:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet("datasets/yellow_tripdata_2015-01.parquet")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [4]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       1| 2015-01-13 11:46:19|  2015-01-13 12:08:42|              2|          2.6|   -74.00537109375|40.737003326416016|         1|    

In [5]:
for column in df.columns:
    null_count = df.filter(col(column).isNull()).count()
    print(column, null_count)

VendorID 0


tpep_pickup_datetime 0
tpep_dropoff_datetime 0
passenger_count 0
trip_distance 0
pickup_longitude 0
pickup_latitude 0
RateCodeID 0
store_and_fwd_flag 0
dropoff_longitude 0
dropoff_latitude 0
payment_type 0
fare_amount 0
extra 0
mta_tax 0
tip_amount 0
tolls_amount 0
improvement_surcharge 3
total_amount 0


In [6]:
df.count()

12748986

In [7]:
df.groupBy("VendorID").count().show()
df.groupBy("VendorID").count().plot.bar(x="VendorID", y="count").show()

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1|6101189|
|       2|6647797|
+--------+-------+



In [8]:
df.groupBy("RateCodeID").count().orderBy("RateCodeID").show()
df.groupBy("RateCodeID").count().plot.bar(x="RateCodeID", y="count").show()

+----------+--------+
|RateCodeID|   count|
+----------+--------+
|         1|12464898|
|         2|  224723|
|         3|   17700|
|         4|    4128|
|         5|   36896|
|         6|     134|
|        99|     507|
+----------+--------+



In [9]:
df.groupBy("store_and_fwd_flag").count().show()
df.groupBy("store_and_fwd_flag").count().plot.bar(
    x="store_and_fwd_flag", y="count"
).show()

+------------------+--------+
|store_and_fwd_flag|   count|
+------------------+--------+
|                 Y|  115033|
|                 N|12633953|
+------------------+--------+



In [10]:
df.groupBy("payment_type").count().orderBy("payment_type").show()
df.groupBy("payment_type").count().plot.bar(x="payment_type", y="count").show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|7881388|
|           2|4816992|
|           3|  38632|
|           4|  11972|
|           5|      2|
+------------+-------+



In [11]:
df.describe(
    [
        "passenger_count",
        "trip_distance",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
    ]
).show()

25/10/23 18:27:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+---------------------+-----------------+
|summary|   passenger_count|     trip_distance|   pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|       fare_amount|             extra|           mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|     total_amount|
+-------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+---------------------+-----------------+
|  count|          12748986|          12748986|           12748986|          12748986|          12748986|          12748986|          12748986|          12748986|          12748986|          12748986|           127489

In [12]:
for column_name in [
    "passenger_count",
    "trip_distance",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
]:
    if column_name == "tolls_amount":
        df.plot.hist(column=column_name, bins=25, title=column_name).show()
        df.plot.box(column=column_name).show()
    else:
        df.plot.hist(column=column_name, title=column_name).show()
        df.plot.box(column=column_name).show()

25/10/23 18:27:40 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:27:40 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:27:42 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:27:48 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:27:48 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:27:50 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:27:55 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:27:55 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:27:57 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:28:01 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:01 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:03 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:28:06 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:06 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:08 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


...

25/10/23 18:28:28 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:28 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:28 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:28:31 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:31 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:33 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:28:36 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:36 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:39 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:28:42 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:42 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:42 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


25/10/23 18:28:45 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:45 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
25/10/23 18:28:47 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.
